# Notebook 01 - Validasi dan kualitas dataset_v05 (TK-3)

Bahan untuk **subbagian 3.1 Dataset** (rubrik kriteria 1, persiapan dan kualitas data).
Notebook ini memeriksa apakah dataset layak dianalisis sebelum deskriptif (Notebook 02), anotasi, sentimen, dan topic modeling.
Sebaran waktu dan konsentrasi per video tidak diulang di sini; keduanya ada di Notebook 02.

Isi:
1. Asal-usul data (run, parameter, versi)
2. Struktur dan tipe kolom
3. Nilai kosong: menurut desain atau tidak terduga
4. Duplikat dan teks berulang
5. Konsistensi relasi dan urutan waktu
6. Validitas nilai
7. Pemeriksaan privasi
8. Dampak filter `eligible`
9. Kualitas teks untuk analisis
10. Cakupan pengambilan komentar dan balasan
11. Tabel isu kualitas data (untuk laporan) dan ringkasan angka

**Etika.** Tidak ada teks komentar, username, atau ID yang ditampilkan. Semua keluaran berupa jumlah dan persentase,
sehingga `outputs/` aman di-commit. Data tetap di `data/processed/` (tidak masuk repo).

**Cara menjalankan.** Sama dengan Notebook 02: buka dari root repo atau dari `notebooks/`. Bila dataset ada di folder lain,
atur variabel lingkungan `DATASET_PATH` ke `dataset_v05.parquet` (file `.metadata.json` harus di folder yang sama).

In [1]:
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():          # dijalankan dari notebooks/
    ROOT = ROOT.parent
DATA = Path(os.environ.get("DATASET_PATH", ROOT / "data" / "processed" / "dataset_v05.parquet"))
TAB = ROOT / "outputs" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

KATA_PANJANG = 8        # teks >= sekian kata yang berulang dari akun berbeda dianggap kandidat salin-tempel
ULANG_MIN = 3           # ... bila muncul minimal sekian kali

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

def angka(x, des=0):
    s = f"{x:,.{des}f}"
    return s.replace(",", "_").replace(".", ",").replace("_", ".")

def persen(x, des=1):
    return angka(100 * x, des) + "%"

def simpan_tabel(tbl, nama, index=True):
    tbl.to_csv(TAB / f"{nama}.csv", index=index, encoding="utf-8")
    print("Disimpan:", f"outputs/tables/{nama}.csv")

R = {}   # ringkasan angka, ditulis di bagian 11

## 0. Memuat data

`dtype_asli` dicatat sebelum konversi, supaya bagian 2 memeriksa tipe sebagaimana tersimpan di file.

In [2]:
df = pd.read_parquet(DATA) if DATA.suffix == ".parquet" else pd.read_csv(DATA, low_memory=False)
dtype_asli = df.dtypes.astype(str)
for c in ("item_id", "parent_id", "conversation_id", "author_hash"):
    df[c] = df[c].astype("string")
for c in ("created_at", "_scraped_at"):
    df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")
if df["eligible"].dtype != bool:
    for c in [c for c in df.columns if c.startswith("flag_")] + ["eligible"]:
        df[c] = df[c].astype(str).str.lower().eq("true")
for c in ("like_count", "reply_count", "share_count", "view_count"):
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("float")

meta = json.loads(DATA.with_name(DATA.name.split(".")[0] + ".metadata.json").read_text(encoding="utf-8"))
PERIODE = meta["keywords_snapshot"]["periode"]
mulai = pd.Timestamp(PERIODE["mulai"], tz="UTC")
akhir = pd.Timestamp(PERIODE["selesai"], tz="UTC") + pd.Timedelta(days=1)   # eksklusif

el = df[df["eligible"]]
kom_el = el[el["item_type"].isin(["comment", "reply"])]
BENDERA = [c for c in df.columns if c.startswith("flag_")]
R["dataset"] = meta["dataset"]
print(f"{meta['dataset']}: {angka(len(df))} record, {angka(len(el))} eligible, {len(df.columns)} kolom.")

dataset_v05: 21.650 record, 18.783 eligible, 31 kolom.


## 1. Asal-usul data

Dataset dibuat oleh `src/process.py` dari run scraping TikTok. Parameter tiap run perlu dilaporkan karena menentukan apa yang
terambil: urutan pencarian, jumlah halaman video, halaman komentar, dan apakah balasan diambil.

In [3]:
run_meta = pd.DataFrame([{
    "run_id": r["run_id"],
    "status": r["status"],
    "mulai_utc": r["started_at"][:16].replace("T", " "),
    "urutan": r["params"].get("tiktok_sort"),
    "halaman_video": r["params"].get("max_pages"),
    "halaman_komentar": r["params"].get("comment_pages"),
    "halaman_balasan": r["params"].get("tiktok_replies"),
    "record_manifest": r["written_manifest"],
} for r in meta["runs_used"]])
run_meta["record_di_dataset"] = run_meta["run_id"].map(df["_run_id"].value_counts()).fillna(0).astype(int)

R["asal_usul"] = {
    "skrip": meta["script"], "commit": meta["git_commit"], "dibuat_utc": meta["created_at"][:19],
    "lingkungan": meta["environment"], "jumlah_run": len(run_meta),
    "run_terputus": run_meta.loc[run_meta["status"] != "completed", "run_id"].tolist(),
    "waktu_pengambilan_utc": [str(df["_scraped_at"].min())[:16], str(df["_scraped_at"].max())[:16]],
    "dedup": meta["parameters"]["dedup_key"],
}
simpan_tabel(run_meta, "val_01_run", index=False)
print("Pengambilan data:", " s/d ".join(R["asal_usul"]["waktu_pengambilan_utc"]), "UTC")
print("Run tidak selesai:", R["asal_usul"]["run_terputus"] or "tidak ada")
print("Peringatan dari process.py:", meta.get("warnings") or "tidak ada")
run_meta

Disimpan: outputs/tables/val_01_run.csv
Pengambilan data: 2026-10-25 07:48 s/d 2026-10-25 12:55 UTC
Run tidak selesai: ['tiktok_20261025_155253']
Peringatan dari process.py: ['tiktok_20261025_155253: status run `interrupted`.']


,run_id,status,mulai_utc,urutan,halaman_video,halaman_komentar,halaman_balasan,record_manifest,record_di_dataset
0,tiktok_20261025_144837,completed,2026-10-25 07:48,likes,3,2,0,10440,10440
1,tiktok_20261025_150459,completed,2026-10-25 08:04,likes,1,1,1,5387,5387
2,tiktok_20261025_155253,interrupted,2026-10-25 08:52,likes,1,1,1,2460,2460
3,tiktok_20261025_190404,completed,2026-10-25 12:04,likes,1,1,1,3363,3363


## 2. Struktur dan tipe kolom

Kolom dicocokkan dengan daftar di `metadata.json` (sumbernya kamus data di `data/README.md`), lalu tipe tiap kolom diperiksa.

In [4]:
diharapkan = meta["columns"]
R["kolom_hilang"] = [c for c in diharapkan if c not in df.columns]
R["kolom_tambahan"] = [c for c in df.columns if c not in diharapkan]

TIPE_HARAPAN = {
    **{c: "teks" for c in ("platform", "item_type", "item_id", "parent_id", "conversation_id", "author_hash",
                           "text", "text_clean", "language", "url", "query", "_run_id")},
    **{c: "waktu" for c in ("created_at", "_scraped_at")},
    **{c: "angka" for c in ("like_count", "reply_count", "share_count", "view_count")},
    **{c: "boolean" for c in BENDERA + ["eligible"]},
    "hashtags": "daftar",
}
def jenis_dtype(c):
    d = dtype_asli[c]
    if d.startswith("datetime"):
        return "waktu"
    if d in ("bool", "boolean"):
        return "boolean"
    if d.startswith(("int", "Int", "float", "Float")):
        return "angka"
    if c == "hashtags":
        contoh = df[c].dropna()
        return "daftar" if len(contoh) and isinstance(contoh.iloc[0], (list, np.ndarray)) else "teks"
    return "teks"

struktur = pd.DataFrame({
    "dtype_tersimpan": dtype_asli,
    "jenis": [jenis_dtype(c) for c in df.columns],
    "jenis_harapan": [TIPE_HARAPAN.get(c, "?") for c in df.columns],
})
struktur["sesuai"] = struktur["jenis"] == struktur["jenis_harapan"]
R["kolom_tipe_tidak_sesuai"] = struktur.index[~struktur["sesuai"]].tolist()
simpan_tabel(struktur.rename_axis("kolom"), "val_02_struktur")
print("Kolom hilang:", R["kolom_hilang"] or "tidak ada", "| kolom tambahan:", R["kolom_tambahan"] or "tidak ada",
      "| tipe tidak sesuai:", R["kolom_tipe_tidak_sesuai"] or "tidak ada")
struktur

Disimpan: outputs/tables/val_02_struktur.csv
Kolom hilang: tidak ada | kolom tambahan: tidak ada | tipe tidak sesuai: tidak ada


,dtype_tersimpan,jenis,jenis_harapan,sesuai
platform,object,teks,teks,True
item_type,object,teks,teks,True
item_id,object,teks,teks,True
parent_id,object,teks,teks,True
conversation_id,object,teks,teks,True
author_hash,object,teks,teks,True
text,object,teks,teks,True
text_clean,object,teks,teks,True
created_at,"datetime64[ns, UTC]",waktu,waktu,True
like_count,Int64,angka,angka,True


## 3. Nilai kosong

Kosong dibedakan dari nol (`like_count = 0` dihitung terisi); teks yang hanya spasi dihitung kosong.
Banyak kolom kosong **menurut desain**: TikTok tidak memberi jumlah bagikan/tayang untuk komentar, balasan tidak punya jumlah balasan,
video tidak punya induk, dan `url` komentar/balasan sengaja dikosongkan karena tautannya membuka profil penulis.
Yang perlu perhatian hanya kosong **tidak terduga**.

In [5]:
KOSONG_DESAIN = {
    ("post", "parent_id"),
    ("comment", "share_count"), ("comment", "view_count"), ("comment", "url"),
    ("reply", "reply_count"), ("reply", "share_count"), ("reply", "view_count"), ("reply", "url"),
}
KOLOM_CEK = ["author_hash", "text", "text_clean", "created_at", "parent_id", "conversation_id", "like_count",
             "reply_count", "share_count", "view_count", "hashtags", "language", "url", "query"]

def kosong(s):
    if s.name == "hashtags":
        return s.isna()
    if str(s.dtype) in ("object", "string"):
        return s.isna() | s.astype("string").str.strip().eq("")
    return s.isna()

baris = []
for t, g in df.groupby("item_type"):
    for c in KOLOM_CEK:
        n = int(kosong(g[c]).sum())
        status = "menurut desain" if (t, c) in KOSONG_DESAIN else ("tidak terduga" if n else "lengkap")
        baris.append({"item_type": t, "kolom": c, "n_kosong": n, "persen_kosong": round(100 * n / len(g), 2), "status": status})
kosong_tbl = pd.DataFrame(baris)
tak_terduga = kosong_tbl[kosong_tbl["status"] == "tidak terduga"]
R["kosong_tidak_terduga"] = {f"{r.item_type}.{r.kolom}": r.n_kosong for r in tak_terduga.itertuples()}
R["teks_kosong"] = int(df["flag_text_missing"].sum())
simpan_tabel(kosong_tbl, "val_03_kosong", index=False)
print("Kosong tidak terduga:", R["kosong_tidak_terduga"] or "tidak ada")
kosong_tbl.pivot(index="kolom", columns="item_type", values="persen_kosong").reindex(KOLOM_CEK)

Disimpan: outputs/tables/val_03_kosong.csv
Kosong tidak terduga: {'comment.text': 117, 'comment.text_clean': 117, 'post.text': 2, 'post.text_clean': 2, 'reply.text': 267, 'reply.text_clean': 267}


item_type,comment,post,reply
kolom,,,
author_hash,0.00,0.00,0.00
text,1.09,1.31,2.47
text_clean,1.09,1.31,2.47
created_at,0.00,0.00,0.00
parent_id,0.00,100.00,0.00
conversation_id,0.00,0.00,0.00
like_count,0.00,0.00,0.00
reply_count,0.00,0.00,100.00
share_count,100.00,0.00,100.00


Teks kosong (termasuk yang hanya berisi penanda media seperti `[Foto]`) sudah ditandai `flag_text_missing` dan dikeluarkan dari analisis.

## 4. Duplikat dan teks berulang

Tiga tingkat:
- **ID ganda**: harus nol karena sudah dideduplikasi dengan kunci `(platform, item_id)`.
- **Komentar ganda dari akun yang sama**: sudah ditandai `flag_repeated_by_author`.
- **Teks sama dari akun berbeda** (di antara komentar+balasan eligible): teks pendek generik seperti ucapan selamat wajar berulang.
  Yang patut dicurigai sebagai salin-tempel atau spam adalah teks panjang (≥ `KATA_PANJANG` kata) yang muncul ≥ `ULANG_MIN` kali.
  Isinya tidak ditampilkan; tinjau manual di laptop bila perlu.

In [6]:
teks_norm = (kom_el["text_clean"].fillna("").str.lower()
             .str.replace(r"\s+", " ", regex=True).str.strip())
ada_huruf = teks_norm.str.contains(r"[a-z]")
frek = teks_norm[ada_huruf].value_counts()
n_kata = pd.Series({t: len(t.split()) for t in frek.index[frek >= 2]}, dtype="float")
panjang_ulang = frek[(frek >= ULANG_MIN).to_numpy() & np.array([len(t.split()) >= KATA_PANJANG for t in frek.index], dtype=bool)]

R["duplikat"] = {
    "id_ganda": int(df.duplicated(["platform", "item_id"]).sum()),
    "komentar_ganda_akun_sama": int(df["flag_repeated_by_author"].sum()),
    "teks_unik_eligible": int(len(frek)),
    "teks_muncul_2x_atau_lebih": int((frek >= 2).sum()),
    "record_pada_teks_berulang": int(frek[frek >= 2].sum()),
    "teks_muncul_10x_atau_lebih": int((frek >= 10).sum()),
    "median_kata_teks_berulang": float(n_kata.median()) if len(n_kata) else None,
    "teks_panjang_berulang": int(len(panjang_ulang)),
    "record_teks_panjang_berulang": int(panjang_ulang.sum()),
}
dup = pd.Series(R["duplikat"], name="nilai", dtype=object).to_frame()
simpan_tabel(dup.rename_axis("indikator"), "val_04_duplikat")
dup

Disimpan: outputs/tables/val_04_duplikat.csv


,nilai
id_ganda,0
komentar_ganda_akun_sama,117
teks_unik_eligible,17306
teks_muncul_2x_atau_lebih,214
record_pada_teks_berulang,648
teks_muncul_10x_atau_lebih,2
median_kata_teks_berulang,1.0
teks_panjang_berulang,1
record_teks_panjang_berulang,3


## 5. Konsistensi relasi dan urutan waktu

Struktur TikTok: video → komentar (`parent_id` = video) → balasan (`parent_id` = komentar); seluruh pohon berbagi `conversation_id` = ID video.
Setiap baris di tabel adalah aturan yang seharusnya tidak pernah dilanggar.

In [7]:
idx = df.set_index("item_id")
post = df[df["item_type"] == "post"]
com = df[df["item_type"] == "comment"]
rep = df[df["item_type"] == "reply"]
id_post, id_com, id_semua = set(post["item_id"]), set(com["item_id"]), set(df["item_id"])

aturan = [
    ("Video tidak punya induk", len(post), int(post["parent_id"].notna().sum())),
    ("conversation_id video = ID video", len(post), int((post["conversation_id"] != post["item_id"]).sum())),
    ("Induk komentar adalah video di dataset", len(com), int((~com["parent_id"].isin(id_post)).sum())),
    ("conversation_id komentar = ID video induk", len(com), int((com["conversation_id"] != com["parent_id"]).sum())),
    ("Induk balasan adalah komentar di dataset", len(rep), int((~rep["parent_id"].isin(id_com)).sum())),
    ("conversation_id balasan = milik komentar induk", len(rep),
     int((rep["conversation_id"] != rep["parent_id"].map(idx["conversation_id"])).sum())),
    ("Komentar tidak lebih awal dari videonya", len(com),
     int((com["created_at"] < com["parent_id"].map(idx["created_at"])).sum())),
    ("Balasan tidak lebih awal dari komentar induknya", len(rep),
     int((rep["created_at"] < rep["parent_id"].map(idx["created_at"])).sum())),
]
relasi = pd.DataFrame(aturan, columns=["aturan", "n_diperiksa", "n_pelanggaran"])
R["pelanggaran_relasi"] = int(relasi["n_pelanggaran"].sum())

# Keterkaitan setelah filter eligible: konteks bisa berada di record yang tidak eligible
el_id = set(el["item_id"])
com_el = kom_el[kom_el["item_type"] == "comment"]
rep_el = kom_el[kom_el["item_type"] == "reply"]
R["komentar_eligible_video_tidak_eligible"] = int((~com_el["parent_id"].isin(el_id)).sum())
R["balasan_eligible_induk_tidak_eligible"] = int((~rep_el["parent_id"].isin(el_id)).sum())
simpan_tabel(relasi, "val_05_relasi", index=False)
print(f"Komentar eligible yang videonya tidak eligible: {angka(R['komentar_eligible_video_tidak_eligible'])}")
print(f"Balasan eligible yang komentar induknya tidak eligible: {angka(R['balasan_eligible_induk_tidak_eligible'])}")
relasi

Disimpan: outputs/tables/val_05_relasi.csv
Komentar eligible yang videonya tidak eligible: 272
Balasan eligible yang komentar induknya tidak eligible: 158


,aturan,n_diperiksa,n_pelanggaran
0,Video tidak punya induk,153,0
1,conversation_id video = ID video,153,0
2,Induk komentar adalah video di dataset,10702,0
3,conversation_id komentar = ID video induk,10702,0
4,Induk balasan adalah komentar di dataset,10795,0
5,conversation_id balasan = milik komentar induk,10795,0
6,Komentar tidak lebih awal dari videonya,10702,0
7,Balasan tidak lebih awal dari komentar induknya,10795,0


Relasi yang rusak membuat utas RQ3 tidak bisa ditelusuri, jadi nol pelanggaran adalah syarat. Dua angka terakhir bukan kesalahan:
filter `eligible` bekerja per record, sehingga sebuah balasan bisa eligible walaupun komentar induknya tidak (mis. induknya berbahasa lain
atau hanya mention). **Untuk RQ3, baca konteks induk dari dataset lengkap**, bukan dari subset eligible.

## 6. Validitas nilai

In [8]:
lang = kom_el["language"].fillna("(kosong)").value_counts()
R["validitas"] = {
    "created_at_tidak_terbaca": int(df["created_at"].isna().sum()),
    "sebelum_periode": int((df["created_at"] < mulai).sum()),
    "sesudah_periode": int((df["created_at"] >= akhir).sum()),
    "created_at_maks": str(df["created_at"].max())[:16],
    "angka_negatif": int(sum((df[c] < 0).sum() for c in ("like_count", "reply_count", "share_count", "view_count"))),
    "author_hash_format_salah": int((~df["author_hash"].fillna("").str.fullmatch(r"[0-9a-f]{16}")).sum()),
    "item_type_tidak_dikenal": int((~df["item_type"].isin(["post", "comment", "reply", "quote"])).sum()),
}
R["bahasa_komentar_balasan_eligible"] = {k: int(v) for k, v in lang.head(6).items()}
R["porsi_kode_id"] = round(float(lang.get("id", 0) / lang.sum()), 4)
validitas = pd.Series(R["validitas"], name="nilai", dtype=object).to_frame()
simpan_tabel(validitas.rename_axis("pemeriksaan"), "val_06_validitas")
bahasa = pd.DataFrame({"jumlah": lang, "porsi": (lang / lang.sum()).round(4)}).head(10)
simpan_tabel(bahasa.rename_axis("kode_bahasa"), "val_06b_bahasa_eligible")
display(validitas)
bahasa

Disimpan: outputs/tables/val_06_validitas.csv
Disimpan: outputs/tables/val_06b_bahasa_eligible.csv


,nilai
created_at_tidak_terbaca,0
sebelum_periode,0
sesudah_periode,1822
created_at_maks,2026-10-25 12:17
angka_negatif,0
author_hash_format_salah,0
item_type_tidak_dikenal,0


,jumlah,porsi
language,,
id,15610,0.8376
ms,1160,0.0622
un,1059,0.0568
en,350,0.0188
tr,70,0.0038
tl,62,0.0033
es,53,0.0028
zh,27,0.0014
pt,22,0.0012


Record di luar periode semuanya **sesudah** periode berakhir: scraper menyaring video menurut tanggal unggah, tetapi komentar pada video
itu masih terus ditulis sampai hari pengambilan. Record ini ditandai `flag_out_of_period`.

Kode bahasa berasal dari deteksi otomatis TikTok dan sering keliru untuk teks pendek. Kode `ms` dan `un` sengaja tidak dikeluarkan karena
umumnya bahasa Indonesia informal; kode asing lain hanya dikeluarkan bila teksnya ≥ 3 kata (lihat `data/README.md`).

## 7. Pemeriksaan privasi

- `author_hash` harus berupa hash, dan `url` komentar/balasan harus kosong.
- Setiap @username di `text` seharusnya sudah tersamar menjadi `@user_<hash>`, dan menjadi `[USER]` di `text_clean`.
  Tanda `@` yang tersisa dikelompokkan menurut polanya; isinya tidak ditampilkan.
- Pola surel dan nomor telepon dihitung sebagai pemeriksaan tambahan.

In [9]:
teks = df["text"].fillna("")
sisa_at = teks.str.findall(r"(.?)@(?!user_)([A-Za-z0-9_.]{2,})")
def pola_at(pasangan):
    pre, tok = pasangan
    if pre == "@":
        return "@@nama (kemungkinan mention lolos)"
    if tok.startswith("."):
        return "@.nama (kemungkinan mention lolos)"
    if pre and (pre.isalnum()):
        return "@ di tengah kata (bukan mention)"
    return "lainnya"
pola = pd.Series([pola_at(p) for daftar in sisa_at for p in daftar], dtype="string").value_counts()
record_sisa = sisa_at.str.len() > 0

R["privasi"] = {
    "teks_dengan_mention_tersamar": int(teks.str.contains("@user_").sum()),
    "text_clean_dengan_[USER]": int(df["text_clean"].fillna("").str.contains(r"\[USER\]").sum()),
    "text_clean_dengan_[URL]": int(df["text_clean"].fillna("").str.contains(r"\[URL\]").sum()),
    "record_dengan_@_tersisa": int(record_sisa.sum()),
    "record_dengan_@_tersisa_eligible": int((record_sisa & df["eligible"]).sum()),
    "pola_@_tersisa": {k: int(v) for k, v in pola.items()},
    "url_komentar_balasan_terisi": int(df.loc[df["item_type"] != "post", "url"].fillna("").ne("").sum()),
    "pola_surel": int(teks.str.contains(r"[\w.+-]+@[\w-]+\.[a-z]{2,}", case=False).sum()),
    "pola_telepon": int(teks.str.contains(r"(?:\+62|\b0)8\d{7,11}\b").sum()),
}
privasi = pd.Series({k: v for k, v in R["privasi"].items() if not isinstance(v, dict)}, name="nilai").to_frame()
simpan_tabel(privasi.rename_axis("pemeriksaan"), "val_07_privasi")
display(privasi)
pola.rename("jumlah_kemunculan").to_frame()

Disimpan: outputs/tables/val_07_privasi.csv


,nilai
teks_dengan_mention_tersamar,793
text_clean_dengan_[USER],816
text_clean_dengan_[URL],12
record_dengan_@_tersisa,11
record_dengan_@_tersisa_eligible,8
url_komentar_balasan_terisi,0
pola_surel,0
pola_telepon,0


,jumlah_kemunculan
@ di tengah kata (bukan mention),5
@.nama (kemungkinan mention lolos),3
@@nama (kemungkinan mention lolos),3


Pola `@@nama` dan `@.nama` lolos dari regex `MENTION` di `src/common/anonymize.py`, karena regex itu menolak `@` yang didahului `@`
dan nama yang diawali titik. Jumlahnya sangat kecil, tetapi tetap berarti ada kemungkinan username yang tidak tersamar.
Penanganan yang diusulkan: record ini tidak dikutip (sejalan dengan aturan parafrase), dan regex diperbaiki bila dataset dibuat ulang.
`@` di tengah kata (mis. pengganti huruf "a") bukan mention.

## 8. Dampak filter `eligible`

Record tidak dihapus; setiap kriteria eksklusi menjadi kolom `flag_*`. Satu record bisa terkena lebih dari satu flag.

In [10]:
per_tipe = df.groupby("item_type")[BENDERA].sum().T
per_tipe = per_tipe[[c for c in ("post", "comment", "reply") if c in per_tipe.columns]]
per_tipe["total"] = per_tipe.sum(axis=1)
per_tipe["sama_dengan_metadata"] = [int(per_tipe.at[f, "total"]) == sum(meta["flags"].get(f, {}).values()) for f in per_tipe.index]
assert per_tipe["sama_dengan_metadata"].all(), "Jumlah flag berbeda dengan metadata"

n_flag = df[BENDERA].sum(axis=1)
alur = pd.DataFrame({
    "dataset": df["item_type"].value_counts(),
    "eligible": el["item_type"].value_counts(),
}).reindex(["post", "comment", "reply"]).fillna(0).astype(int)
alur["dikeluarkan"] = alur["dataset"] - alur["eligible"]
alur["porsi_dikeluarkan"] = (alur["dikeluarkan"] / alur["dataset"]).round(4)
alur.loc["total"] = [alur["dataset"].sum(), alur["eligible"].sum(), alur["dikeluarkan"].sum(),
                     round(alur["dikeluarkan"].sum() / alur["dataset"].sum(), 4)]
alur[["dataset", "eligible", "dikeluarkan"]] = alur[["dataset", "eligible", "dikeluarkan"]].astype(int)

R["eksklusi"] = {
    "masuk": sum(v["masuk"] for v in meta["steps"].values()),
    "dataset": int(len(df)),
    "eligible": int(len(el)),
    "dikeluarkan": int(len(df) - len(el)),
    "porsi_dikeluarkan": round(float(1 - len(el) / len(df)), 4),
    "record_lebih_dari_satu_flag": int((n_flag >= 2).sum()),
    "per_flag": {f: int(per_tipe.at[f, "total"]) for f in per_tipe.index if per_tipe.at[f, "total"] > 0},
}
simpan_tabel(per_tipe.rename_axis("flag"), "val_08a_flag_per_jenis")
simpan_tabel(alur.rename_axis("item_type"), "val_08b_alur_eligible")
display(per_tipe)
alur

Disimpan: outputs/tables/val_08a_flag_per_jenis.csv
Disimpan: outputs/tables/val_08b_alur_eligible.csv


item_type,post,comment,reply,total,sama_dengan_metadata
flag_text_missing,2,119,268,389,True
flag_non_indonesian,4,135,101,240,True
flag_mention_only,0,34,353,387,True
flag_repeated_by_author,0,50,67,117,True
flag_out_of_period,0,1049,773,1822,True
flag_created_at_invalid,0,0,0,0,True
flag_author_not_hashed,0,0,0,0,True
flag_item_type_invalid,0,0,0,0,True
flag_parent_missing,0,0,0,0,True
flag_conversation_missing,0,0,0,0,True


,dataset,eligible,dikeluarkan,porsi_dikeluarkan
item_type,,,,
post,153,147,6,0.0392
comment,10702,9350,1352,0.1263
reply,10795,9286,1509,0.1398
total,21650,18783,2867,0.1324


## 9. Kualitas teks untuk analisis

Komentar TikTok pendek, informal, dan banyak emoji. Angka di bawah (komentar+balasan eligible, dari `text_clean`) menjadi dasar
keputusan prapemrosesan per teknik: model sentimen, BERTopic, dan aturan anotasi untuk komentar emoji saja.

In [11]:
tc = kom_el["text_clean"].fillna("")
kata = tc.str.split().str.len()
huruf = tc.str.contains(r"[^\W\d_]")
emoji = tc.str.contains(r"[\U0001F000-\U0001FAFF☀-➿⬀-⯿]")
R["teks"] = {
    "n": int(len(tc)),
    "median_kata": float(kata.median()),
    "p90_kata": float(kata.quantile(.9)),
    "porsi_satu_kata": round(float((kata == 1).mean()), 4),
    "porsi_maks_dua_kata": round(float((kata <= 2).mean()), 4),
    "porsi_tanpa_huruf": round(float((~huruf).mean()), 4),          # hanya emoji/tanda baca/angka
    "n_tanpa_huruf": int((~huruf).sum()),
    "porsi_mengandung_emoji": round(float(emoji.mean()), 4),
    "porsi_mengandung_[USER]": round(float(tc.str.contains(r"\[USER\]").mean()), 4),
}
per_jenis = pd.DataFrame({
    t: {"median_kata": kata[m].median(), "porsi_maks_dua_kata": (kata[m] <= 2).mean(),
        "porsi_tanpa_huruf": (~huruf[m]).mean(), "porsi_mengandung_emoji": emoji[m].mean()}
    for t, m in (("komentar", kom_el["item_type"] == "comment"), ("balasan", kom_el["item_type"] == "reply"))
}).T.round(4)
simpan_tabel(per_jenis.rename_axis("jenis"), "val_09_kualitas_teks")
print(f"Komentar+balasan tanpa huruf (emoji/tanda baca saja): {angka(R['teks']['n_tanpa_huruf'])} "
      f"({persen(R['teks']['porsi_tanpa_huruf'])}); ≤2 kata: {persen(R['teks']['porsi_maks_dua_kata'])}")
per_jenis

Disimpan: outputs/tables/val_09_kualitas_teks.csv
Komentar+balasan tanpa huruf (emoji/tanda baca saja): 885 (4,8%); ≤2 kata: 22,1%


,median_kata,porsi_maks_dua_kata,porsi_tanpa_huruf,porsi_mengandung_emoji
komentar,6.0,0.1824,0.0413,0.4465
balasan,5.0,0.2599,0.0537,0.3778


## 10. Cakupan pengambilan

Membandingkan yang terkumpul dengan jumlah yang dilaporkan TikTok saat pengambilan:
`reply_count` video = jumlah komentar, `reply_count` komentar = jumlah balasan. Dihitung sebelum filter `eligible`.
Cakupan komentar juga dilaporkan di Notebook 02 dengan rumus yang sama.

Cakupan balasan rendah karena desain pengambilan: run pertama (`halaman_balasan = 0`) tidak mengambil balasan, dan run lain hanya
mengambil satu halaman balasan per komentar. Cakupan tidak dipecah per run karena run berjalan tumpang-tindih dan deduplikasi menyimpan
versi yang paling akhir diambil, sehingga `_run_id` sebuah komentar belum tentu run yang meminta balasannya.

In [12]:
kom_terkumpul = com.groupby("parent_id").size()
cak_kom = pd.concat({"terkumpul": kom_terkumpul, "dilaporkan": post.set_index("item_id")["reply_count"]}, axis=1).dropna()
cak_kom = cak_kom[cak_kom["dilaporkan"] > 0]

bal_terkumpul = rep.groupby("parent_id").size()
cak_bal = pd.concat({"terkumpul": bal_terkumpul, "dilaporkan": com.set_index("item_id")["reply_count"]}, axis=1)
cak_bal["terkumpul"] = cak_bal["terkumpul"].fillna(0)
cak_bal = cak_bal[cak_bal["dilaporkan"] > 0]

R["cakupan"] = {
    "komentar_terkumpul": int(cak_kom["terkumpul"].sum()),
    "komentar_dilaporkan": int(cak_kom["dilaporkan"].sum()),
    "porsi_komentar": round(float(cak_kom["terkumpul"].sum() / cak_kom["dilaporkan"].sum()), 4),
    "median_porsi_komentar_per_video": round(float((cak_kom["terkumpul"] / cak_kom["dilaporkan"]).clip(upper=1).median()), 4),
    "komentar_berbalasan_menurut_platform": int(len(cak_bal)),
    "porsi_komentar_berbalasan_yang_balasannya_terambil": round(float((cak_bal["terkumpul"] > 0).mean()), 4),
    "balasan_terkumpul": int(cak_bal["terkumpul"].sum()),
    "balasan_dilaporkan": int(cak_bal["dilaporkan"].sum()),
    "porsi_balasan": round(float(cak_bal["terkumpul"].sum() / cak_bal["dilaporkan"].sum()), 4),
}
cakupan = pd.Series(R["cakupan"], name="nilai", dtype=object).to_frame()
simpan_tabel(cakupan.rename_axis("indikator"), "val_10_cakupan")
cakupan

Disimpan: outputs/tables/val_10_cakupan.csv


,nilai
komentar_terkumpul,10702
komentar_dilaporkan,222285
porsi_komentar,0.0481
median_porsi_komentar_per_video,0.2459
komentar_berbalasan_menurut_platform,3585
porsi_komentar_berbalasan_yang_balasannya_terambil,0.3844
balasan_terkumpul,10795
balasan_dilaporkan,56210
porsi_balasan,0.192


## 11. Tabel isu kualitas data dan ringkasan angka

Tabel berikut disusun otomatis dari angka di atas, dengan struktur yang sama seperti tabel isu kualitas di TK-2 tetapi memakai angka
dataset_v05. Kolom *penanganan* menyebut apa yang sudah dilakukan pipeline dan apa yang dilakukan di tahap analisis.
Salin ke subbagian Dataset lalu ringkas sesuai batas kata; angka diambil dari `val_ringkasan.json`.

In [13]:
d, v, p, t, c, e = R["duplikat"], R["validitas"], R["privasi"], R["teks"], R["cakupan"], R["eksklusi"]
pf = e["per_flag"]
isu = pd.DataFrame([
    ("Data hilang",
     f"{angka(R['teks_kosong'])} record tanpa teks; kolom lain kosong hanya menurut desain (mis. jumlah tayang/bagikan pada komentar)",
     "Ditandai flag_text_missing dan dikeluarkan; kosong dibedakan dari nol"),
    ("Duplikasi",
     f"{d['id_ganda']} ID ganda; {angka(d['komentar_ganda_akun_sama'])} komentar ganda dari akun yang sama",
     "Deduplikasi (platform, item_id); komentar ganda ditandai, kemunculan pertama dipertahankan"),
    ("Spam dan noise",
     f"{angka(pf.get('flag_mention_only', 0))} record hanya mention; {d['teks_panjang_berulang']} teks panjang berulang dari akun berbeda "
     f"({d['record_teks_panjang_berulang']} record); teks berulang lain umumnya pendek dan generik",
     "Mention saja dikeluarkan; relevansi dan spam dinilai saat anotasi"),
    ("Konsistensi relasi",
     f"{R['pelanggaran_relasi']} pelanggaran relasi induk–anak dan urutan waktu; "
     f"{angka(R['balasan_eligible_induk_tidak_eligible'])} balasan eligible berinduk komentar tidak eligible",
     "Konteks utas RQ3 dibaca dari dataset lengkap"),
    ("Temporal",
     f"{angka(v['sesudah_periode'])} record ditulis sesudah periode (hingga {v['created_at_maks'][:10]}); tidak ada yang sebelum periode",
     "Ditandai flag_out_of_period dan dikeluarkan"),
    ("Bahasa dan teks pendek",
     f"{persen(R['porsi_kode_id'])} komentar+balasan eligible berkode id; {angka(pf.get('flag_non_indonesian', 0))} record asing dikeluarkan; "
     f"{persen(t['porsi_maks_dua_kata'])} berisi ≤2 kata dan {persen(t['porsi_tanpa_huruf'])} tanpa huruf",
     "Kode ms/un tidak dikeluarkan; prapemrosesan diputuskan per teknik; aturan emoji di codebook"),
    ("Cakupan dan bias sampling",
     f"Terambil {persen(c['porsi_komentar'])} komentar dan {persen(c['porsi_balasan'])} balasan dari jumlah yang dilaporkan platform; "
     "pencarian diurutkan menurut suka",
     "Dinyatakan sebagai batasan; klaim dibatasi pada percakapan yang paling terlihat"),
    ("Privasi",
     f"Username dipseudonimkan; URL komentar dikosongkan; {sum(n for k, n in p['pola_@_tersisa'].items() if 'lolos' in k)} kemunculan "
     "pola @ yang kemungkinan mention lolos penyamaran",
     "Data hanya di Drive privat; tanpa kutipan verbatim; regex penyamaran diperbaiki bila dataset dibuat ulang"),
], columns=["isu", "temuan", "penanganan"])
simpan_tabel(isu, "val_11_isu_kualitas", index=False)
(TAB / "val_11_isu_kualitas.md").write_text(
    "| Isu | Temuan | Penanganan |\n|---|---|---|\n" +
    "\n".join(f"| {r.isu} | {r.temuan} | {r.penanganan} |" for r in isu.itertuples()) + "\n", encoding="utf-8")
print("Disimpan: outputs/tables/val_11_isu_kualitas.md (siap tempel)")
isu

Disimpan: outputs/tables/val_11_isu_kualitas.csv
Disimpan: outputs/tables/val_11_isu_kualitas.md (siap tempel)


,isu,temuan,penanganan
0,Data hilang,389 record tanpa teks; kolom lain kosong hanya...,Ditandai flag_text_missing dan dikeluarkan; ko...
1,Duplikasi,0 ID ganda; 117 komentar ganda dari akun yang ...,"Deduplikasi (platform, item_id); komentar gand..."
2,Spam dan noise,387 record hanya mention; 1 teks panjang berul...,Mention saja dikeluarkan; relevansi dan spam d...
3,Konsistensi relasi,0 pelanggaran relasi induk–anak dan urutan wak...,Konteks utas RQ3 dibaca dari dataset lengkap
4,Temporal,1.822 record ditulis sesudah periode (hingga 2...,Ditandai flag_out_of_period dan dikeluarkan
5,Bahasa dan teks pendek,"83,8% komentar+balasan eligible berkode id; 24...",Kode ms/un tidak dikeluarkan; prapemrosesan di...
6,Cakupan dan bias sampling,"Terambil 4,8% komentar dan 19,2% balasan dari ...",Dinyatakan sebagai batasan; klaim dibatasi pad...
7,Privasi,Username dipseudonimkan; URL komentar dikosong...,Data hanya di Drive privat; tanpa kutipan verb...


In [14]:
def ke_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if np.isnan(o) else round(float(o), 4)
    if isinstance(o, dict):
        return {str(k): ke_json(x) for k, x in o.items()}
    if isinstance(o, (list, tuple)):
        return [ke_json(x) for x in o]
    return o

R["dibuat_dari"] = DATA.name
(TAB / "val_ringkasan.json").write_text(json.dumps(ke_json(R), ensure_ascii=False, indent=2), encoding="utf-8")
print("Disimpan: outputs/tables/val_ringkasan.json")

Disimpan: outputs/tables/val_ringkasan.json


## Catatan untuk penulisan subbagian Dataset

- **Pisahkan pengumpulan dari persiapan.** Prosedur scraping cukup diringkas dengan merujuk TK-2; fokus 3.1 adalah struktur data,
  langkah persiapan (`process.py`: dedup, `text_clean`, flag eksklusi), dan kualitas data.
- **Alur jumlah data** (masuk → dataset → eligible per jenis item) diambil dari `val_08b_alur_eligible.csv`.
- **Nol pelanggaran relasi** layak disebut satu kalimat, karena itulah yang memungkinkan utas RQ3 ditelusuri.
- **Cakupan** adalah isu kualitas terbesar dan perlu masuk batasan, bukan hanya tabel.
- **Netral.** Isu kualitas menggambarkan data dan cara pengambilannya, bukan pengguna atau figur.